# Latent-space probe results — exploratory plots

Loads every `predictions.csv` under `output/latent_probes/` and the aggregated `summary.csv`.
Headline plot: violin of per-row residuals (predicted − true, transformed space) per probe, split by baseline.

Baselines:
- **none** — real latents.
- **gaussian** — iid N(0,1) features of identical shape (capacity-of-MLP floor).
- **shuffle** — real latents, rows permuted (signal floor; preserves latent distribution / inter-dim correlations).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

ROOT = Path('.').resolve()
PROBES   = ['flux_skew', 'flux_kurt', 'lit_prot', 'tars_prot', 'num_flares', 'total_ed']
BASELINES = ['none', 'gaussian', 'shuffle']
BASELINE_COLOR = {'none': '#2c7fb8', 'gaussian': '#c0c0c0', 'shuffle': '#8a8a8a'}

def run_dir(probe, baseline):
    return ROOT / (probe if baseline == 'none' else f'{probe}__{baseline}')

summary = pd.read_csv(ROOT / 'summary.csv')
summary

In [ ]:
rows = []
for probe in PROBES:
    for baseline in BASELINES:
        p = run_dir(probe, baseline) / 'predictions.csv'
        if not p.exists():
            print(f'missing: {p}')
            continue
        df = pd.read_csv(p)
        df['probe'] = probe
        df['baseline'] = baseline
        df['residual'] = df['y_pred_transformed'] - df['y_true_transformed']
        df['abs_residual'] = df['residual'].abs()
        rows.append(df)
preds = pd.concat(rows, ignore_index=True)
print(f'{len(preds):,} rows across {preds["probe"].nunique()} probes x {preds["baseline"].nunique()} baselines')
preds.head()

In [ ]:
# Pearson r per probe x baseline (sanity check vs summary.md).
fig, ax = plt.subplots(figsize=(10, 4.5))
width = 0.27
x = np.arange(len(PROBES))
for i, b in enumerate(BASELINES):
    rs = [float(summary[(summary.probe == p) & (summary.baseline == b)]['r'].iloc[0])
          if not summary[(summary.probe == p) & (summary.baseline == b)].empty else np.nan
          for p in PROBES]
    ax.bar(x + (i - 1) * width, rs, width, label=b, color=BASELINE_COLOR[b])
ax.axhline(0, color='k', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(PROBES, rotation=20, ha='right')
ax.set_ylabel('Pearson r (transformed space)')
ax.set_title('Recovery of summary statistics from the 1536-D latent space')
ax.legend(title='baseline', frameon=False)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()

In [ ]:
# Violin: residual distribution per probe, split by baseline.
# Residual lives in the transformed space the model trained in (asinh / log10),
# so units differ across probes -> one subplot per probe.

CLIP_PCTL = 99.5  # trim extreme outliers for readability; set to 100 for full tails

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=False)
for ax, probe in zip(axes.ravel(), PROBES):
    data, labels, colors = [], [], []
    for b in BASELINES:
        r = preds[(preds.probe == probe) & (preds.baseline == b)]['residual'].to_numpy()
        if CLIP_PCTL < 100:
            lim = np.nanpercentile(np.abs(r), CLIP_PCTL)
            r = r[np.abs(r) <= lim]
        data.append(r); labels.append(b); colors.append(BASELINE_COLOR[b])
    parts = ax.violinplot(data, showmeans=False, showmedians=True, widths=0.85)
    for body, c in zip(parts['bodies'], colors):
        body.set_facecolor(c); body.set_edgecolor('k'); body.set_alpha(0.7)
    ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.6)
    ax.set_xticks(np.arange(1, len(BASELINES) + 1)); ax.set_xticklabels(labels)
    transform = summary[summary.probe == probe]['transform'].iloc[0]
    ax.set_title(f'{probe}  ({transform})')
    ax.set_ylabel('residual (pred − true)')
    ax.grid(axis='y', alpha=0.3)
fig.suptitle(f'Per-row residuals by baseline (clipped at {CLIP_PCTL}th pct of |residual|)', y=1.02)
fig.tight_layout()

In [ ]:
# Alt view: |residual| on a log y-axis -> all six probes on one panel.
# Useful for eyeballing the real-vs-baseline gap at a glance.
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(12, 4.5))
positions, tick_pos, tick_labels = [], [], []
for i, probe in enumerate(PROBES):
    for j, b in enumerate(BASELINES):
        positions.append(i * 4 + j)
    tick_pos.append(i * 4 + 1); tick_labels.append(probe)

data, colors = [], []
for probe in PROBES:
    for b in BASELINES:
        r = preds[(preds.probe == probe) & (preds.baseline == b)]['abs_residual'].to_numpy()
        r = r[r > 0]
        data.append(r); colors.append(BASELINE_COLOR[b])

parts = ax.violinplot(data, positions=positions, widths=0.9, showmedians=True)
for body, c in zip(parts['bodies'], colors):
    body.set_facecolor(c); body.set_edgecolor('k'); body.set_alpha(0.75)
ax.set_yscale('log')
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels, rotation=15, ha='right')
ax.set_ylabel('|residual| (transformed space, log scale)')
ax.set_title('Absolute residual: real (blue) vs gaussian / shuffle (grey)')
ax.grid(axis='y', alpha=0.3, which='both')
ax.legend(handles=[Patch(color=BASELINE_COLOR[b], label=b) for b in BASELINES],
          frameon=False, loc='upper right')
fig.tight_layout()

In [ ]:
# Scratch cell - explore from here.
# preds columns: GaiaDR3_ID, sector, fold, y_true_raw, y_pred_raw,
#                y_true_transformed, y_pred_transformed, probe, baseline, residual, abs_residual

preds.groupby(['probe', 'baseline'])['abs_residual'].describe(percentiles=[0.5, 0.95])[['count', 'mean', '50%', '95%']]